In [ ]:
import segmentation_models_pytorch as smp
from torchinfo import summary

model = smp.DPT(
    encoder_name="tu-nextvit_base",
    encoder_depth=4,
    classes=6,
    decoder_readout="ignore",
)
summary(model, input_size=(1, 3, 512, 512))

Layer (type:depth-idx)                                            Output Shape              Param #
DPT                                                               [1, 6, 512, 512]          --
├─TimmViTEncoder: 1-1                                             [1, 96, 128, 128]         --
│    └─NextViT: 2-1                                               --                        1,027,048
│    │    └─Sequential: 3-1                                       [1, 64, 128, 128]         75,904
│    │    └─Sequential: 3-2                                       --                        43,715,904
├─DPTDecoder: 1-2                                                 [1, 256, 256, 256]        --
│    └─ModuleList: 2-8                                            --                        --
│    │    └─ReadoutIgnoreBlock: 3-3                               [1, 96, 128, 128]         --
│    └─ModuleList: 2-9                                            --                        (recursive)
│    │    └─Reass

In [14]:
model = smp.Segformer(
    encoder_name="mit_b4",
    encoder_depth=5,
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,
)
summary(model, input_size=(1, 3, 512, 512))

Layer (type:depth-idx)                        Output Shape              Param #
Segformer                                     [1, 6, 512, 512]          --
├─MixVisionTransformerEncoder: 1-1            [1, 3, 512, 512]          --
│    └─OverlapPatchEmbed: 2-1                 [1, 64, 128, 128]         --
│    │    └─Conv2d: 3-1                       [1, 64, 128, 128]         9,472
│    │    └─LayerNorm: 3-2                    [1, 64, 128, 128]         128
│    └─Sequential: 2-2                        [1, 64, 128, 128]         --
│    │    └─Block: 3-3                        [1, 64, 128, 128]         314,880
│    │    └─Block: 3-4                        [1, 64, 128, 128]         314,880
│    │    └─Block: 3-5                        [1, 64, 128, 128]         314,880
│    └─LayerNorm: 2-3                         [1, 64, 128, 128]         128
│    └─OverlapPatchEmbed: 2-4                 [1, 128, 64, 64]          --
│    │    └─Conv2d: 3-6                       [1, 128, 64, 64]          73,

In [ ]:
model_kwargs:
  encoder_name: mit_b4
  encoder_depth: 5
  encoder_weights: imagenet
  in_channels: *in_channels
  classes: *n_classes

In [17]:
model = smp.UPerNet(encoder_name="tu-convnext_small", in_channels=3, classes=6)
summary(model, input_size=(1, 3, 1024, 1024))

Layer (type:depth-idx)                                  Output Shape              Param #
UPerNet                                                 [1, 6, 1024, 1024]        --
├─TimmUniversalEncoder: 1-1                             [1, 3, 1024, 1024]        --
│    └─FeatureListNet: 2-1                              [1, 96, 256, 256]         --
│    │    └─Conv2d: 3-1                                 [1, 96, 256, 256]         4,704
│    │    └─LayerNorm2d: 3-2                            [1, 96, 256, 256]         192
│    │    └─ConvNeXtStage: 3-3                          [1, 96, 256, 256]         237,888
│    │    └─ConvNeXtStage: 3-4                          [1, 192, 128, 128]        992,256
│    │    └─ConvNeXtStage: 3-5                          [1, 384, 64, 64]          32,747,520
│    │    └─ConvNeXtStage: 3-6                          [1, 768, 32, 32]          15,470,592
├─UPerNetDecoder: 1-2                                   [1, 256, 256, 256]        --
│    └─ModuleList: 2-2        

In [ ]:

model_kwargs:

  encoder_name: tu-convnext_small
  encoder_depth: 5
  encoder_weights: imagenet
  in_channels: *in_channels
  classes: *n_classes

In [20]:
import torch
from transformers import UperNetForSemanticSegmentation

# 1. Initialize the model
model = UperNetForSemanticSegmentation.from_pretrained(
    "openmmlab/upernet-swin-tiny", num_labels=6, ignore_mismatched_sizes=True
)

# Put model in evaluation mode (disables dropout, etc.)
model.eval()

# 2. Define your batch size and spatial dimensions
batch_size = 2
channels = 3
height = 1024  # You can change this to 512 as well
width = 1024  # You can change this to 512 as well

# 3. Create dummy random X (images) and y (ground truth segmentation masks)
# X shape: [batch_size, channels, height, width]
X = torch.randn(batch_size, channels, height, width)

# y shape: [batch_size, height, width] -> Values between 0 and num_labels-1 (5)
y = torch.randint(low=0, high=6, size=(batch_size, height, width), dtype=torch.long)

# 4. Run the forward pass
# We wrap it in torch.no_grad() since we are just doing a dry-run check
with torch.no_grad():
    outputs = model(pixel_values=X, labels=y)

# 5. Inspect the outputs
print("Loss:", outputs.loss.item())
print("Logits Shape:", outputs.logits.shape)
# Logits shape will be [batch_size, num_labels, height, width] -> [2, 6, 256, 256]

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/307 [00:00<?, ?it/s]

[transformers] UperNetForSemanticSegmentation LOAD REPORT from: openmmlab/upernet-swin-tiny
Key                              | Status   |                                                                                                     
---------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
auxiliary_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])
decode_head.classifier.weight    | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 512, 1, 1]) vs model:torch.Size([6, 512, 1, 1])
auxiliary_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      

Notes:
- MISMATCH:	ckpt w

Loss: 2.5729994773864746
Logits Shape: torch.Size([2, 6, 1024, 1024])
